# W10 — Assignment notebook


Cada estudiante entrega:
1. Evidencia de particionamiento
2. Captura o texto del número de archivos.
3. Resumen por partición.
4. Evidencia de pruning
5. Archivo w10b_explain_analyze_pruning.txt.
6. Explicación corta de qué filtro usó.
7. Decisión de partición
8. ¿Por qué disc_era sí o no?
9. ¿Qué otra columna evaluaría?
10. ¿Qué riesgo de small files encontró?
11. Reflexión breve
12. ¿Cuándo particionar ayuda?
13. ¿Cuándo particionar empeora el diseño?

In [ ]:
from pathlib import Path
import duckdb, os
#os.chdir("..")
print("cwd:", os.getcwd())

PROJECT_ROOT = Path(".").resolve()
DB_PATH      = PROJECT_ROOT / "data" / "exoplanets.duckdb"
RAW_CSV      = PROJECT_ROOT / "data" / "raw" / "pscomppars.csv"
PART_DIR     = PROJECT_ROOT / "artifacts" / "silver_partitioned"

con = duckdb.connect(str(DB_PATH))
def sql_path(p): return "'" + p.resolve().as_posix().replace("'", "''") + "'"

con.execute("DROP VIEW IF EXISTS raw_ps")
con.execute(f"CREATE VIEW raw_ps AS SELECT * FROM read_csv_auto({sql_path(RAW_CSV)})")

# Asegura que silver_planet_v3 existe (o recréala)
con.execute("DROP TABLE IF EXISTS silver_planet_v3")
con.execute("""
CREATE TABLE silver_planet_v3 AS
SELECT
    pl_name, hostname,
    LOWER(TRIM(hostname)) AS hostname_canon,
    discoverymethod,
    LOWER(TRIM(discoverymethod)) AS discoverymethod_canon,
    disc_year,
    TRY_CAST(disc_year AS INTEGER) AS disc_year_int,
    CASE
        WHEN TRY_CAST(disc_year AS INTEGER) IS NULL THEN true
        WHEN TRY_CAST(disc_year AS INTEGER) < 1980  THEN true
        WHEN TRY_CAST(disc_year AS INTEGER) > 2026  THEN true
        ELSE false
    END AS disc_year_bad,
    CASE
        WHEN TRY_CAST(disc_year AS INTEGER) < 2000 THEN 'pre-2000'
        WHEN TRY_CAST(disc_year AS INTEGER) < 2010 THEN '2000s'
        WHEN TRY_CAST(disc_year AS INTEGER) < 2020 THEN '2010s'
        ELSE '2020s'
    END AS disc_era,
    pl_orbper, pl_rade, pl_bmasse, pl_eqt, sy_dist, ra, dec
FROM raw_ps
WHERE pl_name IS NOT NULL AND hostname IS NOT NULL
  AND (pl_rade   IS NULL OR (pl_rade > 0 AND pl_rade <= 30))
  AND (pl_bmasse IS NULL OR pl_bmasse > 0)
""")

# Particionar por disc_era — 1 CSV por era
PART_DIR.mkdir(parents=True, exist_ok=True)
eras = con.execute("SELECT DISTINCT disc_era FROM silver_planet_v3 WHERE disc_era IS NOT NULL ORDER BY disc_era").fetchall()
for (era,) in eras:
    out = PART_DIR / f"disc_era={era}" / "data.csv"
    out.parent.mkdir(parents=True, exist_ok=True)
    con.execute(f"""
        COPY (SELECT * FROM silver_planet_v3 WHERE disc_era = '{era}')
        TO '{out.as_posix()}' (FORMAT CSV, HEADER TRUE)
    """)
    n = con.execute(f"SELECT COUNT(*) FROM silver_planet_v3 WHERE disc_era = '{era}'").fetchone()[0]
    print(f"  disc_era={era} → {n} filas → {out}")

# Contar archivos
files = list(PART_DIR.rglob("*.csv"))
print(f"\nTotal archivos generados: {len(files)}")
for f in sorted(files):
    print(f"  {f.relative_to(PROJECT_ROOT)}")

cwd: c:\Users\nancy\Documents\Ingenieria_datos
  disc_era=2000s → 378 filas → C:\Users\nancy\Documents\Ingenieria_datos\artifacts\silver_partitioned\disc_era=2000s\data.csv
  disc_era=2010s → 3681 filas → C:\Users\nancy\Documents\Ingenieria_datos\artifacts\silver_partitioned\disc_era=2010s\data.csv
  disc_era=2020s → 2012 filas → C:\Users\nancy\Documents\Ingenieria_datos\artifacts\silver_partitioned\disc_era=2020s\data.csv
  disc_era=pre-2000 → 30 filas → C:\Users\nancy\Documents\Ingenieria_datos\artifacts\silver_partitioned\disc_era=pre-2000\data.csv

Total archivos generados: 4
  artifacts\silver_partitioned\disc_era=2000s\data.csv
  artifacts\silver_partitioned\disc_era=2010s\data.csv
  artifacts\silver_partitioned\disc_era=2020s\data.csv
  artifacts\silver_partitioned\disc_era=pre-2000\data.csv


In [3]:
# Resumen por partición
con.sql("""
    SELECT disc_era, COUNT(*) AS n_planets,
           MIN(disc_year) AS primer_anio,
           MAX(disc_year) AS ultimo_anio
    FROM silver_planet_v3
    WHERE disc_era IS NOT NULL
    GROUP BY disc_era
    ORDER BY disc_era
""").show()

# EXPLAIN ANALYZE — evidencia de pruning (solo lee particiones filtradas)
plan = con.execute("""
    EXPLAIN ANALYZE
    SELECT COUNT(*), AVG(pl_rade)
    FROM silver_planet_v3
    WHERE disc_era = '2010s'
""").fetchdf()

# Guardar como txt (entregable #5)
plan_txt = plan.to_string(index=False)
Path("artifacts/w10b_explain_analyze_pruning.txt").write_text(plan_txt, encoding="utf-8")
print(plan_txt[:2000])

┌──────────┬───────────┬─────────────┬─────────────┐
│ disc_era │ n_planets │ primer_anio │ ultimo_anio │
│ varchar  │   int64   │    int64    │    int64    │
├──────────┼───────────┼─────────────┼─────────────┤
│ 2000s    │       378 │        2000 │        2009 │
│ 2010s    │      3681 │        2010 │        2019 │
│ 2020s    │      2012 │        2020 │        2026 │
│ pre-2000 │        30 │        1992 │        1999 │
└──────────┴───────────┴─────────────┴─────────────┘

  explain_key                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             